# 02. Feature Engineering & Preprocessing Pipeline
**Author**: Member 2 (Feature Engineer)  
**Project**: Smart Loan Default & Credit Risk Assessment System  
**Module**: Machine Learning Module - Group Project Assignment

This notebook implements the **6 mandatory feature engineering techniques** required by Section 5 of the assignment.

## 1. Load Cleaned Dataset
We load the verified cleaned dataset provided by Member 1 from `ml/data/processed/cleaned_credit_data.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ml.src.features import DomainFeatureCreator, IQRWinsorizer, build_full_preprocessing_pipeline

data_path = os.path.join('..', 'data', 'processed', 'cleaned_credit_data.csv')
df = pd.read_csv(data_path)
print(f'Loaded cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns.')
df.head()

## 2. Implement & Inspect Domain Interaction Features
* **Technique 1**: `debt_burden` (Total loan obligation relative to gross income)
* **Technique 2**: `credit_maturity` (Credit history length relative to legal adult years)
* **Technique 3**: Non-linear `log1p` transformation on `person_income` and `loan_amnt`
* **Technique 5**: Life-stage `age_group` discretization

In [ ]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

creator = DomainFeatureCreator()
X_engineered = creator.fit_transform(X)

print('New features engineered: [debt_burden, credit_maturity, log_income, log_loan_amnt, age_group]')

# Compare skewness
print(f'Raw Income Skewness:      {X["person_income"].skew():.3f} -> Log Income Skewness:      {X_engineered["log_income"].skew():.3f}')
print(f'Raw Loan Amount Skewness: {X["loan_amnt"].skew():.3f} -> Log Loan Amount Skewness: {X_engineered["log_loan_amnt"].skew():.3f}')

X_engineered[['debt_burden', 'credit_maturity', 'log_income', 'log_loan_amnt', 'age_group']].describe()

## 3. Technique 4: Outlier Treatment via IQR Winsorization
Visualizing the bounding logic of `IQRWinsorizer` to cap extreme variance without dropping valid test rows.

In [ ]:
winsorizer = IQRWinsorizer(columns=['debt_burden', 'log_income'])
winsorizer.fit(X_engineered)
print('Fitted IQR Bounds strictly on input training vectors:')
for col, bounds in winsorizer.bounds_.items():
    print(f'  {col}: Lower = {bounds[0]:.4f}, Upper = {bounds[1]:.4f}')

## 4. Full Pipeline Assembly & Execution
Testing the unified `ColumnTransformer` (Ordinal encoding for `loan_grade`, One-Hot for nominals, Standard scaling).

In [ ]:
pipeline = build_full_preprocessing_pipeline()
X_final = pipeline.fit_transform(X)

print(f'Final Transformed Output Shape: {X_final.shape}')
print(f'Contains any NaN values: {np.isnan(X_final).any()}')
print('First transformed vector sample (1x22):')
print(np.round(X_final[0], 3))